# 🧬 Biyolojide Yapay Zeka Dersi - Vize Projesi
# Preimplantasyon Genetik Taramada (PGT-A) Kromozomal Kopya Sayısı (CNV) Profillerinin Yapay Zeka ile Sınıflandırılması

**Ders:** Biyolojide Yapay Zeka (*Artificial Intelligence in Biology - Bio-AI*)  
**Alan:** Genomik Veri Analizi, Hesaplamalı Biyoloji & Tüp Bebek (IVF)  
**Programlama Dili:** Python 3  

---

## 📌 1. Biyolojik Problem Tanımı ve Klinik Önem
Tüp bebek (In Vitro Fertilization - IVF) tedavilerinde başarısız implantasyonun ve erken dönem düşüklerin en temel nedeni **embriyonik anöploididir (kromozom sayı anomalileri)**. Özellikle ileri anne yaşında mayotik ayrılmama (*non-disjunction*) sebebiyle embriyolarda trizomi (örn. Trizomi 21 - Down Sendromu, Trizomi 18 - Edwards Sendromu) veya monozomi (örn. Monozomi X - Turner Sendromu) sıklığı katlanarak artar.

Modern üreme tıbbında bu anomalileri tespit etmek amacıyla 5. gün blastokist evresindeki embriyolardan trofektoderm biyopsisi alınarak **PGT-A (Preimplantasyon Genetik Test - Anöploidi)** uygulanır. PGT-A'da altın standart yöntem **Düşük Kapsamlı Tüm Genom Dizilemedir (Low-Pass NGS ~0.01x - 0.1x)**.

### 🤖 Yapay Zeka Neden Gereklidir?
1. **Düşük Kapsamlı Dizileme Gürültüsü (NGS Noise):** Biyopsiden yalnızca 5-10 hücre elde edildiği için DNA amplifikasyonu (WGA) ve dizileme sinyali oldukça gürültülüdür.
2. **Mozaizm Belirsizliği:** Embriyodaki hücrelerin bir kısmı normal (2n), bir kısmı anöploid olduğunda ara kopya sayı oranları ortaya çıkar (%30-%50 arası sapmalar). İnsan gözüyle bu sınırları ayırt etmek zordur.
3. **Otomatik Teşhis:** Yapay zeka ve çoklu makine öğrenmesi modelleri 24 insan kromozomunun okuma derinliği oranlarını ve z-skorlarını analiz ederek embriyoları **Öploid (Sağlıklı)**, **Anöploid (Anormal)** ve **Mozaik** olarak yüksek kesinlikle sınıflandırır.

## 2. Gerekli Kütüphanelerin Yüklenmesi ve İçe Aktarılması

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

print("✅ Gerekli kütüphaneler başarıyla içe aktarıldı!")

## 3. Biyolojik PGT-A / NGS Veri Simülasyonu
İnsan kromozomları (22 otozom + X, Y) için düşük kapsamlı NGS okuma oranları, anne yaşı dağılımı ve blastokist biyopsi günü parametreleri gerçek klinik frekanslara göre modellenir:
- **Öploid (0):** Normal 2n diploid, oran ~ 1.00
- **Anöploid (1):** Trizomi (+1 kopya -> oran ~ 1.50) veya Monozomi (-1 kopya -> oran ~ 0.50)
- **Mozaik (2):** Parsiyel kopya değişimi (oran ~ 1.25 veya 0.75)

In [ ]:
CHROMOSOMES = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
AUTOSOMES = [f"chr{i}" for i in range(1, 23)]
CLASS_NAMES = ["Euploid (Öploid)", "Aneuploid (Anöploid)", "Mosaic (Mozaik)"]

def generate_pgta_dataset(n_samples: int = 650, random_seed: int = 42) -> pd.DataFrame:
    np.random.seed(random_seed)
    data = []

    n_euploid = int(n_samples * 0.50)
    n_aneuploid = int(n_samples * 0.35)
    n_mosaic = n_samples - n_euploid - n_aneuploid
    sample_id = 1

    # 1. Öploid Embriyolar
    for _ in range(n_euploid):
        maternal_age = int(np.random.normal(loc=31, scale=3.5))
        maternal_age = max(24, min(42, maternal_age))
        is_female = np.random.rand() > 0.5
        biopsy_day = np.random.choice([5, 6], p=[0.75, 0.25])
        row = {"sample_id": f"EMB_{sample_id:04d}", "maternal_age": maternal_age, "biopsy_day": biopsy_day, "sex": "XX" if is_female else "XY"}
        for chrom in AUTOSOMES:
            row[f"ratio_{chrom}"] = np.random.normal(loc=1.00, scale=0.045)
        if is_female:
            row["ratio_chrX"] = np.random.normal(loc=1.00, scale=0.045)
            row["ratio_chrY"] = np.random.normal(loc=0.02, scale=0.015)
        else:
            row["ratio_chrX"] = np.random.normal(loc=0.50, scale=0.035)
            row["ratio_chrY"] = np.random.normal(loc=0.50, scale=0.035)
        row["target_class"] = 0
        data.append(row)
        sample_id += 1

    # 2. Anöploid Embriyolar
    common_aneuploids = ["chr21", "chr16", "chr18", "chr15", "chr22", "chr13", "chrX"]
    for _ in range(n_aneuploid):
        maternal_age = int(np.random.normal(loc=37, scale=3.8))
        maternal_age = max(28, min(45, maternal_age))
        is_female = np.random.rand() > 0.5
        biopsy_day = np.random.choice([5, 6], p=[0.55, 0.45])
        row = {"sample_id": f"EMB_{sample_id:04d}", "maternal_age": maternal_age, "biopsy_day": biopsy_day, "sex": "XX" if is_female else "XY"}
        for chrom in AUTOSOMES:
            row[f"ratio_{chrom}"] = np.random.normal(loc=1.00, scale=0.050)
        if is_female:
            row["ratio_chrX"] = np.random.normal(loc=1.00, scale=0.050)
            row["ratio_chrY"] = np.random.normal(loc=0.02, scale=0.015)
        else:
            row["ratio_chrX"] = np.random.normal(loc=0.50, scale=0.040)
            row["ratio_chrY"] = np.random.normal(loc=0.50, scale=0.040)
        affected = np.random.choice(common_aneuploids)
        is_trisomy = np.random.rand() > 0.35
        if affected == "chrX":
            row["ratio_chrX"] = np.random.normal(0.50, 0.04) if not is_trisomy and is_female else np.random.normal(1.50, 0.05)
        else:
            row[f"ratio_{affected}"] = np.random.normal(1.50, 0.055) if is_trisomy else np.random.normal(0.50, 0.045)
        row["target_class"] = 1
        data.append(row)
        sample_id += 1

    # 3. Mozaik Embriyolar
    for _ in range(n_mosaic):
        maternal_age = int(np.random.normal(loc=34, scale=3.5))
        maternal_age = max(26, min(43, maternal_age))
        is_female = np.random.rand() > 0.5
        biopsy_day = np.random.choice([5, 6], p=[0.60, 0.40])
        row = {"sample_id": f"EMB_{sample_id:04d}", "maternal_age": maternal_age, "biopsy_day": biopsy_day, "sex": "XX" if is_female else "XY"}
        for chrom in AUTOSOMES:
            row[f"ratio_{chrom}"] = np.random.normal(loc=1.00, scale=0.048)
        if is_female:
            row["ratio_chrX"] = np.random.normal(loc=1.00, scale=0.048)
            row["ratio_chrY"] = np.random.normal(loc=0.02, scale=0.015)
        else:
            row["ratio_chrX"] = np.random.normal(loc=0.50, scale=0.038)
            row["ratio_chrY"] = np.random.normal(loc=0.50, scale=0.038)
        affected = np.random.choice(common_aneuploids[:5])
        row[f"ratio_{affected}"] = np.random.normal(1.26, 0.05) if np.random.rand() > 0.5 else np.random.normal(0.74, 0.045)
        row["target_class"] = 2
        data.append(row)
        sample_id += 1

    return pd.DataFrame(data).sample(frac=1.0, random_state=random_seed).reset_index(drop=True)

raw_df = generate_pgta_dataset(650)
print(f"Veri Seti: {raw_df.shape[0]} embriyo biyopsisi simüle edildi.")
raw_df.head()

## 4. Biyoinformatik ve QC Öznitelik Mühendisliği
Klinik genomik analizlerde kullanılan parametreler türetilir:
- **Kromozom Z-Skorları:** $Z_i = \frac{\text{Oran}_i - 1.0}{\sigma_{\text{baseline}}}$
- **MAPD (Median Absolute Pairwise Difference):** Dizileme gürültü indeksi
- **Maksimum Mutlak Z-Skoru:** En anormal kromozom sapması
- **Anöploidi Yükü (Aneuploidy Load):** Anormal kromozom sayısı

In [ ]:
def engineer_genomic_features(df: pd.DataFrame) -> pd.DataFrame:
    df_feat = df.copy()
    baseline_std = 0.05
    for chrom in AUTOSOMES:
        df_feat[f"zscore_{chrom}"] = (df_feat[f"ratio_{chrom}"] - 1.0) / baseline_std

    autosome_cols = [f"ratio_{chrom}" for chrom in AUTOSOMES]
    ratios_matrix = df_feat[autosome_cols].values
    pairwise_diffs = np.abs(np.diff(ratios_matrix, axis=1))
    df_feat["mapd_noise"] = np.median(pairwise_diffs, axis=1)

    zscore_cols = [f"zscore_{chrom}" for chrom in AUTOSOMES]
    df_feat["max_abs_zscore"] = np.max(np.abs(df_feat[zscore_cols].values), axis=1)
    df_feat["aneuploidy_load"] = np.sum(np.abs(ratios_matrix - 1.0) > 0.15, axis=1)
    df_feat["autosome_variance"] = np.var(ratios_matrix, axis=1)
    return df_feat

df_featured = engineer_genomic_features(raw_df)
feature_cols = [c for c in df_featured.columns if c not in ["sample_id", "sex", "target_class"]]
print(f"Toplam Çıkarılan Biyoinformatik Öznitelik Sayısı: {len(feature_cols)}")

## 5. Çoklu Makine Öğrenmesi Modellerinin Eğitimi ve Çapraz Doğrulama
Aşağıdaki 4 farklı mimari karşılaştırılmaktadır:
1. **Random Forest Classifier (Ağaç Tabanlı Topluluk)**
2. **Support Vector Machine (RBF Çekirdeği)**
3. **HistGradientBoosting (Gradient Artırma)**
4. **Multi-Layer Perceptron (Yapay Sinir Ağı)**

In [ ]:
X = df_featured[feature_cols].values
y = df_featured["target_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Random Forest Classifier": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
    "Support Vector Machine (RBF)": SVC(kernel='rbf', C=2.0, probability=True, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=150, random_state=42),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600, random_state=42)
}

print("=== 5-KATLI ÇAPRAZ DOĞRULAMA (5-FOLD MACRO F1) ===")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1_macro')
    print(f"• {name:<30}: Macro F1 = {scores.mean():.4f} (± {scores.std():.4f})")

## 6. Bağımsız Test Kümesinde Model Karşılaştırması ve Metrik Tablosu

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Macro Precision": prec,
        "Macro Recall": rec,
        "Macro F1": f1,
        "Macro ROC-AUC": auc
    })

metrics_df = pd.DataFrame(results).sort_values(by="Macro ROC-AUC", ascending=False)
display(metrics_df)

best_model_name = metrics_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"\n★ En Başarılı Model: {best_model_name}")
print("\n--- Sınıflandırma Raporu (Classification Report) ---")
print(classification_report(y_test, best_model.predict(X_test_scaled), target_names=CLASS_NAMES))

## 7. Biyoinformatik Görselleştirme: Karyotip CNV Profili, Karışıklık Matrisi, ROC ve Özellik Önemi

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
plt.subplots_adjust(wspace=0.25, hspace=0.32)

# 1. Karışıklık Matrisi
cm = confusion_matrix(y_test, best_model.predict(X_test_scaled))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=["Euploid", "Aneuploid", "Mosaic"],
            yticklabels=["Euploid", "Aneuploid", "Mosaic"])
axes[0, 0].set_title(f"Karışıklık Matrisi ({best_model_name.split()[0]})", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Yapay Zeka Tahmini")
axes[0, 0].set_ylabel("Gerçek Klinik Durum")

# 2. Sınıf Bazında ROC-AUC Eğrileri
y_prob_best = best_model.predict_proba(X_test_scaled)
for class_idx, class_label in enumerate(["Euploid", "Aneuploid", "Mosaic"]):
    y_true_binary = (y_test == class_idx).astype(int)
    fpr, tpr, _ = roc_curve(y_true_binary, y_prob_best[:, class_idx])
    auc_val = roc_auc_score(y_true_binary, y_prob_best[:, class_idx])
    axes[0, 1].plot(fpr, tpr, lw=2, label=f"{class_label} (AUC = {auc_val:.3f})")

axes[0, 1].plot([0, 1], [0, 1], color='navy', linestyle='--', label='Rastgele')
axes[0, 1].set_title("Sınıf Bazında ROC-AUC Eğrileri", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate (Recall)")
axes[0, 1].legend(loc="lower right")
axes[0, 1].grid(alpha=0.3)

# 3. Random Forest Genomik Özellik Önemi
rf_model = trained_models["Random Forest Classifier"]
importances = rf_model.feature_importances_
top_indices = np.argsort(importances)[-12:]

axes[1, 0].barh(range(len(top_indices)), importances[top_indices], color='#2b8cbe')
axes[1, 0].set_yticks(range(len(top_indices)))
axes[1, 0].set_yticklabels([feature_cols[i] for i in top_indices])
axes[1, 0].set_xlabel("Gini Önem Düzeyi")
axes[1, 0].set_title("En Belirleyici 12 Genomik & Klinik Özellik", fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. In Silico 22 Otozom CNV Karyotip Profili
sample_eu = raw_df[raw_df["target_class"] == 0].iloc[0]
sample_an = raw_df[raw_df["target_class"] == 1].iloc[0]
sample_mo = raw_df[raw_df["target_class"] == 2].iloc[0]
x_vals = np.arange(len(AUTOSOMES))

axes[1, 1].plot(x_vals, [sample_eu[f"ratio_{c}"] for c in AUTOSOMES], 'o-', color='#2ca02c', label="Öploid (2n Normal)")
axes[1, 1].plot(x_vals, [sample_an[f"ratio_{c}"] for c in AUTOSOMES], 's-', color='#d62728', label="Anöploid (Trizomi/Monozomi)")
axes[1, 1].plot(x_vals, [sample_mo[f"ratio_{c}"] for c in AUTOSOMES], '^-', color='#ff7f0e', label="Mozaik (Parsiyel)")
axes[1, 1].axhline(1.0, color='black', linestyle=':', label="Diploid Referans (1.0)")
axes[1, 1].axhline(1.5, color='red', linestyle='--', alpha=0.5, label="Trizomi Eşiği (1.5)")
axes[1, 1].axhline(0.5, color='blue', linestyle='--', alpha=0.5, label="Monozomi Eşiği (0.5)")
axes[1, 1].set_xticks(x_vals)
axes[1, 1].set_xticklabels([c.replace("chr", "") for c in AUTOSOMES], fontsize=8)
axes[1, 1].set_xlabel("Kromozom Numarası (1-22)")
axes[1, 1].set_ylabel("Normalize Okuma Oranı (Copy Ratio)")
axes[1, 1].set_title("Örnek Embriyoların 24 Kromozom CNV Profili", fontsize=12, fontweight='bold')
axes[1, 1].legend(loc="upper right", fontsize=8)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Klinik Vaka Tahmin Gösterimi (Inference)

In [ ]:
def test_single_embryo(vaka_adi, anne_yasi, biopsy_day, sex, anomali=None):
    sample = {"sample_id": "TEST", "maternal_age": anne_yasi, "biopsy_day": biopsy_day, "sex": sex}
    for chrom in AUTOSOMES:
        sample[f"ratio_{chrom}"] = np.random.normal(1.0, 0.04)
    sample["ratio_chrX"] = 1.0 if sex == "XX" else 0.5
    sample["ratio_chrY"] = 0.02 if sex == "XX" else 0.5
    if anomali:
        sample[f"ratio_{anomali[0]}"] = anomali[1]
        
    df_s = engineer_genomic_features(pd.DataFrame([sample]))
    vec = np.array([df_s[c].values[0] for c in feature_cols]).reshape(1, -1)
    scaled = scaler.transform(vec)
    
    probs = best_model.predict_proba(scaled)[0]
    pred = np.argmax(probs)
    
    print(f"● {vaka_adi}")
    print(f"  Yapay Zeka Teşhisi : >>> {CLASS_NAMES[pred]} <<< (Güven: %{probs[pred]*100:.2f})")
    print(f"  Olasılıklar        : Öploid: %{probs[0]*100:.1f} | Anöploid: %{probs[1]*100:.1f} | Mozaik: %{probs[2]*100:.1f}\n")

test_single_embryo("Vaka 1: Sağlıklı 5. Gün Dişi Embriyo (46,XX)", 29, 5, "XX")
test_single_embryo("Vaka 2: Down Sendromlu Erkek Embriyo (47,XY,+21)", 39, 6, "XY", anomali=("chr21", 1.52))
test_single_embryo("Vaka 3: Turner Sendromlu Embriyo (45,X)", 34, 5, "XX", anomali=("chrX", 0.48))
test_single_embryo("Vaka 4: Mozaik Trizomi 16 Embriyosu (%40 Mozaik)", 36, 6, "XX", anomali=("chr16", 1.27))